In [8]:
import jieba
from gensim import corpora, models, similarities


In [6]:
jieba.load_userdict('data/userdict_mb.txt')
stopwords = [line.strip() for line in open('data/stopwords.txt', 'r', encoding='utf-8').readlines()]

### 读取文档进行分词

In [3]:
def readfile2wordslist(file_path):
    words_list = []
    cut_words_list = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            line = line.strip()
            if not line:
                continue
            words = jieba.cut(line)
            words = [word for word in words if word not in stopwords and word != ' ']
            words_list.append(line)
            cut_words_list.append(words)
    return words_list, cut_words_list

### 构建tfidf语料库

In [9]:
def calculate_tfidf(file_path):
    words_list,cut_words_list = readfile2wordslist(file_path)
    dictionary = corpora.Dictionary(cut_words_list)
    num_features = len(dictionary.token2id)
    print(f"Number of features: {num_features}")
    words_bag = [dictionary.doc2bow(words) for words in cut_words_list]
    tfidf_model = models.TfidfModel(words_bag)
    tfidf_texts = tfidf_model[words_bag]
    similarity_index = similarities.SparseMatrixSimilarity(tfidf_texts, num_features=num_features)
    return dictionary, tfidf_model, similarity_index, words_list


### 对输入一句话进行分词

In [10]:
def preprocess_text(text):
    # 分词
    words = jieba.cut(text)
    # 去除停用词
    words = [word for word in words if word not in stopwords and word != ' ']
    return words

### 计算相似度

In [11]:
def calculate_similarity(input_text, dictionary, tfidf_model, similarity_index, words_list):
    input_words = preprocess_text(input_text)
    input_bow = dictionary.doc2bow(input_words)
    input_tfidf = tfidf_model[input_bow]
    similarities = similarity_index.get_similarities(input_tfidf)
    sorted_similarities = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)
    top_similarities = sorted_similarities[:5]
    results = [(words_list[i], score) for i, score in top_similarities]
    return results

In [13]:
dictionary, tfidf_model, similarity_index, words_list = calculate_tfidf('data/mb.txt')

Number of features: 548


In [ ]:
input_text = "Apple iPhone 7 （A1660） 128G 黑色 移动联通电信4G手机"
results = calculate_similarity(input_text, dictionary, tfidf_model, similarity_index, words_list)
for result in results:
    print(f"Text: {result[0]}, Similarity Score: {result[1]}") 

Text: Apple iPhone 7 （A1660） 128G 黑色 移动联通电信4G手机, Similarity Score: 1.0
Text: Apple iPhone 7 Plus （A1661） 128G 黑色 移动联通电信4G手机, Similarity Score: 0.6529167294502258
Text: Apple iPhone 6 32GB 金色 移动联通电信4G手机, Similarity Score: 0.44514673948287964
Text: Apple iPhone 6s Plus （A1699） 128G 玫瑰金色 移动联通电信4G手机, Similarity Score: 0.43065381050109863
Text: Apple iPhone 12（A2404） 128GB 黑色 支持移动联通电信5G 双卡双待手机, Similarity Score: 0.40702927112579346
